In [ ]:
import optuna
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
    plot_slice,
    plot_contour,
    plot_rank
)

import warnings
warnings.filterwarnings("ignore")

def apply_latex_style(fig):
    fig.update_layout(
        template="simple_white",
        font=dict(
            family="Computer Modern",
            size=14
        ),
        title=dict(x=0.5, xanchor="center"),
        margin=dict(l=70, r=30, t=70, b=60),
        legend=dict(bgcolor="rgba(255,255,255,0.8)", borderwidth=0)
    )
    
    fig.update_xaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgray", zeroline=False)
    fig.update_yaxes(showgrid=True, gridwidth=0.5, gridcolor="lightgray", zeroline=False)
    return fig

SOP_INSTANCES = [
    "br17.10.sop", "br17.12.sop",
    "ESC07.sop", "ESC12.sop", "ESC25.sop", "ESC47.sop", "ESC63.sop", "ESC78.sop",
    "ft53.1.sop", "ft53.2.sop", "ft53.3.sop", "ft53.4.sop",
    "ft70.1.sop", "ft70.2.sop", "ft70.3.sop", "ft70.4.sop",
    "kro124p.1.sop", "kro124p.2.sop", "kro124p.3.sop",
    "p43.1.sop", "p43.2.sop", "p43.3.sop", "p43.4.sop",
    "prob.42.sop",
    "ry48p.1.sop", "ry48p.2.sop", "ry48p.3.sop", "ry48p.4.sop"
]

print("=" * 50)
print("INSTANCE SELECTION FOR VIEWING")
print("=" * 50)

for i, name in enumerate(SOP_INSTANCES):
    print(f"[{i+1:02d}] {name:<15}", end="")
    if (i + 1) % 4 == 0:
        print()

print()
print() 

input = input("Enter the instance number: ")
choice = int(input)

file_name = SOP_INSTANCES[choice - 1]
key_name = file_name.replace(".sop", "")
db_name = f"./Results/optuna_results_{key_name}.db"
study_name = f"study_{key_name}"
db_url = f"sqlite:///{db_name}"

print(f"\nLoading study '{study_name}' from '{db_name}'...")

study = optuna.load_study(
    study_name=study_name, 
    storage=db_url
)

print(f"Loaded trials: {len(study.trials)}")
print(f"Best value: {study.best_value}")
print(f"Best parameters: {study.best_params}")

/home/kerollan/miniconda3/envs/rl/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INSTANCE SELECTION FOR VIEWING
[01] br17.10.sop    [02] br17.12.sop    [03] ESC07.sop      [04] ESC12.sop      
[05] ESC25.sop      [06] ESC47.sop      [07] ESC63.sop      [08] ESC78.sop      
[09] ft53.1.sop     [10] ft53.2.sop     [11] ft53.3.sop     [12] ft53.4.sop     
[13] ft70.1.sop     [14] ft70.2.sop     [15] ft70.3.sop     [16] ft70.4.sop     
[17] kro124p.1.sop  [18] kro124p.2.sop  [19] kro124p.3.sop  [20] p43.1.sop      
[21] p43.2.sop      [22] p43.3.sop      [23] p43.4.sop      [24] prob.42.sop    
[25] ry48p.1.sop    [26] ry48p.2.sop    [27] ry48p.3.sop    [28] ry48p.4.sop    



Loading study 'study_ry48p.1' from 'optuna_results_ry48p.1.db'...
Loaded trials: 500
Best value: 17258.0
Best parameters: {'alpha': 0.56, 'gamma': 0.15, 'epsilon': 0.05}


### Optimization History

In [2]:
fig_hist = plot_optimization_history(study)

fig_hist.add_hline(
    y=study.best_value,
    line_dash="dash",
    line_color="black"
)

fig_hist = apply_latex_style(fig_hist)

fig_hist.update_layout(
    xaxis_title="Trial",
    yaxis_title="Objective Value"
)

fig_hist

### Hyperparameter Importance (fANOVA)

In [3]:
fig_importance = plot_param_importances(study)

fig_importance = apply_latex_style(fig_importance)

fig_importance.update_yaxes(showgrid=False)

fig_importance.update_layout(
    title="Hyperparameter Importance (fANOVA)",
    xaxis_title="Relative Importance",
)

fig_importance

### Parallel Coordinates

In [4]:
fig_parallel = plot_parallel_coordinate(study)

fig_parallel.update_traces(
    line=dict(
        colorscale="RdBu",
        showscale=True,
        colorbar=dict(
            title="Objective Value",
            ticks="outside"
        )
    )
)

fig_parallel = apply_latex_style(fig_parallel)

fig_parallel.update_layout(
    margin=dict(l=80, r=140, t=80, b=100)
)

fig_parallel

### Slice Plot

In [5]:
fig_slice = plot_slice(study)

fig_slice.update_traces(
    selector=dict(type="scatter"),
    marker=dict(
        colorscale="RdBu",
        showscale=True,
        colorbar=dict(
            title="Trial"
        )
    )
)

fig_slice = apply_latex_style(fig_slice)

fig_slice

### Contour Plot

In [6]:
fig_contour = plot_contour(study)

fig_contour.update_traces(
    selector=dict(type="contour"),
    contours=dict(coloring="heatmap"),
    colorscale="RdBu"
)

fig_contour = apply_latex_style(fig_contour)

fig_contour

### Rank Plot

In [7]:
fig_rank = plot_rank(study)

fig_rank = apply_latex_style(fig_rank)

fig_rank

# Export to PDF

In [8]:
from pathlib import Path
from pypdf import PdfWriter, PdfReader

output_dir = Path("figures_pdf")
output_dir.mkdir(exist_ok=True)

figures = [
    ("01_optimization_history", fig_hist),
    ("02_param_importance", fig_importance),
    ("03_parallel_coordinates", fig_parallel),
    ("04_slice", fig_slice),
    ("05_contour", fig_contour),
    ("06_rank", fig_rank),
]

pdf_paths = []

for name, fig in figures:
    pdf_path = output_dir / f"{name}.pdf"
    fig.write_image(
        pdf_path,
        format="pdf",
        engine="kaleido",
        scale=2
    )
    pdf_paths.append(pdf_path)

writer = PdfWriter()

for pdf in pdf_paths:
    reader = PdfReader(str(pdf))
    for page in reader.pages:
        writer.add_page(page)

final_pdf = output_dir / "optuna_results_all_figures.pdf"

with open(final_pdf, "wb") as f:
    writer.write(f)

print(f"Final PDF generated in: {final_pdf.resolve()}")

Final PDF generated in: /home/kerollan/SOP/figures_pdf/optuna_results_all_figures.pdf
